# Brain Age Prediction and MRI Site Harmonization

**AI for Medicine — Final Project**
Prof. Stefano Diciotti — University of Bologna

**Author:** Alessandro Capialbi

---

**Goal.** Predict chronological brain age from regional cortical thickness (CT) and fractal
dimension (FD) features extracted from T1-weighted MRI across a 36-site, 1,740-subject
multicenter dataset, and quantify how much of the prediction's cross-site generalization gap
is explained by scanner/site effects — and how much of it is recovered by **ComBat
harmonization** (fit globally, with a transparent discussion of the resulting data-leakage
trade-offs — see Section 8 — since ComBat does not support a genuinely out-of-sample
application to an unseen site).

**Data.** Marzi, Giannelli, Barucci, Tessa, Mascalchi, Diciotti. *Efficacy of MRI data
harmonization in the age of machine learning: a multicenter study across 36 datasets.*
Scientific Data (2024). https://doi.org/10.1038/s41597-023-02421-6

- Part I: https://zenodo.org/records/7845311 (ABIDE I/II, FCP-ICBM, CoRR-NKI2 — 1,189 subjects)
- Part II: https://zenodo.org/records/7845361 (IXI — 551 subjects)

**Report template sections this notebook feeds directly:**
`4. Dataset Description`, `5. Data Preprocessing`, `6. Avoiding Data Leakage`,
`7. Machine Learning Pipeline`, `8. Results`, `9. Discussion`.


## 1. Setup and Reproducibility

All randomness is controlled by a single seed. Every model used below (ElasticNet, Random Forest, XGBoost) has a closed-form or deterministic fitting procedure given a fixed seed — unlike deep learning models, there is no GPU-induced non-determinism to worry about here (see course discussion on reproducibility in AI for Medicine).

In [ ]:
# Run this once per Colab session.
# xgboost is upgraded (not pinned) to stay compatible with Colab's preinstalled
# scikit-learn version (xgboost's sklearn-tags API changed with sklearn >= 1.6).
!pip install -q neuroHarmonize==2.4.4 neuroCombat umap-learn==0.5.6
!pip install -q -U xgboost


In [ ]:
import os
import sys
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

# Set to True to re-run the full nested Leave-One-Site-Out cross-validation from scratch
# (36 sites x 3 models x 2 harmonization conditions -- takes a while).
# Set to False to load precomputed results from results/ (fast walkthrough / grading).
RUN_FULL_TRAINING = True

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Python:", sys.version.split()[0])
for pkg in ["numpy", "pandas", "sklearn", "xgboost"]:
    mod = __import__(pkg)
    print(pkg, getattr(mod, "__version__", "?"))


## 2. Data Acquisition (Part I + Part II from Zenodo)

Data are downloaded directly from Zenodo (not redistributed in the repo). Both files share an identical column schema, so the merge is a simple concatenation after a sanity check.

In [ ]:
PART1_URL = "https://zenodo.org/records/7845311/files/multicenter_CT-FD_features_1.csv?download=1"
PART2_URL = "https://zenodo.org/records/7845361/files/multicenter_CT-FD_features_2.csv?download=1"

os.makedirs("data", exist_ok=True)
!wget -q -O data/part1.csv "{PART1_URL}"
!wget -q -O data/part2.csv "{PART2_URL}"

part1 = pd.read_csv("data/part1.csv")
part2 = pd.read_csv("data/part2.csv")

assert list(part1.columns) == list(part2.columns), "Column schema mismatch between Part I and Part II!"

df = pd.concat([part1, part2], ignore_index=True)

print("Part I  :", part1.shape)
print("Part II :", part2.shape)
print("Combined:", df.shape)
assert df["Subject"].duplicated().sum() == 0, "Duplicate subject IDs across parts!"
assert df.isna().sum().sum() == 0, "Unexpected missing values!"
print("No duplicate subjects, no missing values -- confirmed.")
df.head()


## 3. Exploratory Data Analysis — Understanding the Site Effect

Before modeling anything, we need to see: (a) how many sites/subjects we have, (b) whether the age ranges of different sites overlap enough for harmonization to be statistically identifiable, and (c) whether there is a visible "batch effect" in the raw feature space.

In [ ]:
site_summary = (
    df.groupby("SITE")
      .agg(n=("Subject", "size"), age_min=("Age", "min"), age_max=("Age", "max"),
           age_mean=("Age", "mean"), pct_female=("Sex", "mean"))
      .sort_values("n", ascending=False)
)
print(f"Number of distinct sites: {df['SITE'].nunique()}")
site_summary.style.format({"age_min": "{:.1f}", "age_max": "{:.1f}",
                            "age_mean": "{:.1f}", "pct_female": "{:.0%}"})


In [ ]:
# Age range per site -- visualizes the overlap (or lack of it) between sites.
order = site_summary.index
fig, ax = plt.subplots(figsize=(9, 11))
for i, site in enumerate(order):
    row = site_summary.loc[site]
    ax.plot([row.age_min, row.age_max], [i, i], color="steelblue", lw=3, alpha=0.8)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=7)
ax.set_xlabel("Age (years)")
ax.set_title("Age range per site — checking overlap across sites\n"
             "(harmonization needs age overlap to separate site effect from age effect)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/age_range_per_site.png", dpi=150)
plt.show()


In [ ]:
# Batch effect, quick visual check: boxplot of a global feature (cortex_CT) for the
# 12 most populous sites (all 36 would be unreadable).
top_sites = site_summary.head(12).index
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df[df.SITE.isin(top_sites)], x="SITE", y="cortex_CT", order=top_sites, ax=ax)
ax.set_title("Whole-cortex CT by site (12 most populous sites) — visible site-to-site shift")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/cortex_CT_by_site_boxplot.png", dpi=150)
plt.show()


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

FEATURES = [c for c in df.columns if c.endswith("_CT") or c.endswith("_FD")]
print(f"Number of CT/FD features: {len(FEATURES)}")

X_all = StandardScaler().fit_transform(df[FEATURES].values)
pcs = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_all)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(pcs[:, 0], pcs[:, 1], c=df["SITE"].astype("category").cat.codes,
                 cmap="tab20", s=12, alpha=0.7)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("PCA of raw CT/FD features, colored by acquisition site\n"
             "Clustering by color (site) = evidence of a batch/site effect")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/pca_raw_by_site.png", dpi=150)
plt.show()


## 4. Feature Set and Target Definition

- **Features (X):** 11 CT + 11 FD regional measures (whole cortex, left/right, 4 lobes x 2 hemispheres) + `Sex` as a predictor.
- **Target (y):** `Age`.
- **Group variable:** `SITE`, used for Leave-One-Site-Out cross-validation and as the batch variable for harmonization.

In [ ]:
MODEL_FEATURES = FEATURES + ["Sex"]

X_cols = MODEL_FEATURES
y_col = "Age"
group_col = "SITE"

print(f"Model input features ({len(X_cols)}):")
print(X_cols)


## 5. Baseline: Quantifying the Site Effect (Site Classifier, Pre-Harmonization)

If a classifier can predict *which site* a subject was scanned at just from CT/FD values, that is direct quantitative evidence of a batch effect. We use stratified cross-validation (this is a diagnostic on the raw features, not part of the leakage-sensitive age-prediction pipeline).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, f1_score

def site_classifier_cv(X, sites, random_state=RANDOM_STATE):
    clf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                  random_state=random_state, n_jobs=-1)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    y_pred = cross_val_predict(clf, X, sites, cv=skf, n_jobs=-1)
    bal_acc = balanced_accuracy_score(sites, y_pred)
    macro_f1 = f1_score(sites, y_pred, average="macro")
    return bal_acc, macro_f1

X_raw = df[FEATURES].values
sites = df["SITE"].values

bal_acc_pre, macro_f1_pre = site_classifier_cv(X_raw, sites)
n_classes = df["SITE"].nunique()
print(f"Site classifier on RAW features -- balanced accuracy: {bal_acc_pre:.3f}  "
      f"(chance level ~= {1/n_classes:.3f}) | macro-F1: {macro_f1_pre:.3f}")


## 6. Modeling Utilities: Models, Nested Tuning, LOSO-CV Runner

**Avoiding data leakage (report Section 6):**
- **Predictive model (the actual ML task being evaluated):** Leave-One-Site-Out cross-validation
  — the held-out site is never used to fit or tune the age-prediction model. Hyperparameter
  tuning (`RandomizedSearchCV`) also happens *only* within the training fold, so the held-out
  site never influences model selection either.
- **Harmonization (a preprocessing step, not the predictive model):** ComBat is fit once,
  globally, on the full cohort, rather than per-fold and applied out-of-sample — a genuinely
  out-of-sample application to an *unseen* site is not supported by ComBat's own math (see
  Section 8 for the concrete failure this caused and the standard workaround used instead).
  We document this transparently as the one place where the pipeline departs from a strictly
  leakage-free design, and discuss its (mild) implications in the report.

*Simplification, documented for transparency:* the inner CV is a plain `KFold` on the
training pool (which already spans multiple sites), not a grouped CV. A stricter version
would also group the inner folds by site; we note this as a possible refinement in the
Discussion / Future Work.

In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

N_ITER_SEARCH = 20
INNER_CV = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def get_model_space():
    return {
        "ElasticNet": (
            ElasticNet(max_iter=20000, random_state=RANDOM_STATE),
            {"alpha": np.logspace(-3, 1, 30), "l1_ratio": np.linspace(0.05, 0.95, 19)},
        ),
        "RandomForest": (
            RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
            {"n_estimators": [100, 200, 300], "max_depth": [3, 5, 8, 12, None],
             "min_samples_leaf": [1, 2, 4, 8], "max_features": ["sqrt", 0.5, 1.0]},
        ),
        "XGBoost": (
            XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, objective="reg:squarederror"),
            {"n_estimators": [100, 200, 300], "max_depth": [2, 3, 4, 6],
             "learning_rate": [0.01, 0.03, 0.05, 0.1],
             "subsample": [0.7, 0.85, 1.0], "colsample_bytree": [0.7, 0.85, 1.0]},
        ),
    }

def tune_and_fit(estimator, param_dist, X_train, y_train):
    search = RandomizedSearchCV(
        estimator, param_distributions=param_dist, n_iter=N_ITER_SEARCH,
        cv=INNER_CV, scoring="neg_mean_absolute_error",
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_params_


In [ ]:
from neuroHarmonize import harmonizationLearn, harmonizationApply


def run_loso_cv(data, feature_cols=None, X_source=None, target_col="Age", verbose=True):
    # Leave-One-Site-Out CV for age regression.
    #
    # Pass `feature_cols` to use raw columns from `data` directly. Pass `X_source` (a full
    # feature matrix aligned with `data`'s row order -- e.g. a globally ComBat-harmonized
    # matrix, see Section 8) to use it instead.
    #
    # Why harmonization is not re-fit inside each fold: ComBat's `harmonizationApply` is
    # designed to add new subjects to an ALREADY-KNOWN site/batch, not to harmonize a
    # genuinely unseen site -- which is exactly the held-out fold in LOSO-CV. Attempting it
    # raises an IndexError (the fitted design matrix has no column for a batch the model
    # never saw). This is a real architectural limitation of ComBat, not specific to our
    # code -- see the Section 8 markdown for the full discussion and its consequence for
    # data leakage.
    model_space = get_model_space()
    fold_rows, pred_rows = [], []

    site_arr = data["SITE"].values
    y_full = data[target_col].values
    subj_full = data["Subject"].values
    X_full = X_source if X_source is not None else data[feature_cols].values.astype(float)
    harmonized = X_source is not None

    site_list = pd.unique(site_arr)
    for i, held_out in enumerate(site_list):
        train_mask = site_arr != held_out
        test_mask = ~train_mask

        X_train_use, X_test_use = X_full[train_mask], X_full[test_mask]
        y_train, y_test = y_full[train_mask], y_full[test_mask]
        test_subjects = subj_full[test_mask]

        for model_name, (estimator, param_dist) in model_space.items():
            best_model, best_params = tune_and_fit(estimator, param_dist, X_train_use, y_train)
            y_pred = best_model.predict(X_test_use)

            mae = mean_absolute_error(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            r2 = r2_score(y_test, y_pred) if len(y_test) > 1 else np.nan

            fold_rows.append({"site": held_out, "n_test": len(y_test), "model": model_name,
                               "mae": mae, "rmse": rmse, "r2": r2, "harmonized": harmonized})
            for subj, ta, pa in zip(test_subjects, y_test, y_pred):
                pred_rows.append({"site": held_out, "model": model_name, "harmonized": harmonized,
                                   "subject": subj, "true_age": ta, "pred_age": pa})

        if verbose:
            print(f"[{i+1:2d}/{len(site_list)}] site={held_out:22s} n_test={len(y_test):4d}  done")

    return pd.DataFrame(fold_rows), pd.DataFrame(pred_rows)


## 7. Age Prediction — LOSO-CV WITHOUT Harmonization (Baseline)

**Note on R² in LOSO-CV.** The per-site R² averaged in the table below can be strongly
negative for sites with a narrow internal age range (e.g. an 8-13 y/o site), because
`r2_score` compares errors against *that site's own* small variance, not the global one —
this is a statistical artifact of averaging R² across heterogeneous groups, not a modeling
error. MAE/RMSE remain reliable throughout. As a more stable aggregate figure we also report
a **pooled R²**, computed on all out-of-fold predictions concatenated together, immediately
after each results table.

In [ ]:
RAW_FOLDS_PATH = f"{RESULTS_DIR}/loso_raw_folds.csv"
RAW_PREDS_PATH = f"{RESULTS_DIR}/loso_raw_preds.csv"

if RUN_FULL_TRAINING or not os.path.exists(RAW_FOLDS_PATH):
    folds_raw, preds_raw = run_loso_cv(df, feature_cols=MODEL_FEATURES)
    folds_raw.to_csv(RAW_FOLDS_PATH, index=False)
    preds_raw.to_csv(RAW_PREDS_PATH, index=False)
else:
    folds_raw = pd.read_csv(RAW_FOLDS_PATH)
    preds_raw = pd.read_csv(RAW_PREDS_PATH)

folds_raw.groupby("model")[["mae", "rmse", "r2"]].mean().round(2)


In [ ]:
# R^2 computed on all out-of-fold predictions pooled together (standard for LOSO-CV),
# instead of averaging per-site R^2 (unstable when a held-out site has near-zero internal
# age variance -- see markdown note above).
def pooled_r2(preds_df, model_name):
    sub = preds_df[preds_df.model == model_name]
    return r2_score(sub["true_age"], sub["pred_age"])

print("Pooled R^2 (all out-of-fold LOSO predictions combined), RAW features:")
for m in preds_raw["model"].unique():
    print(f"  {m:15s} R^2 = {pooled_r2(preds_raw, m):.3f}")


## 8. ComBat Harmonization + Age Prediction WITH Harmonization

**Methodological note (important — ties into report Section 6, "Avoiding Data Leakage").**
The original design fit ComBat *inside* each LOSO fold, learning batch parameters only on the
training sites and applying them out-of-sample to the held-out site. In practice this fails:
`neuroHarmonize`'s out-of-sample `harmonizationApply` is built to add *new subjects to an
already-known site* (e.g. new patients scanned later at a hospital that was part of the
original harmonization), not to handle a *genuinely unseen site* — which is exactly what
Leave-One-Site-Out requires for the held-out fold. Attempting it raises an `IndexError` (the
fitted design matrix has no batch column for a site the model never saw). This is a real,
documented limitation of ComBat-style harmonization, not a bug in our pipeline — harmonizing
data from a scanner with zero prior reference subjects is an open problem in the field.

**Practical approach used below** (standard in the harmonization literature, including studies
using this exact dataset): ComBat is fit **once, globally**, on the full 1,740-subject cohort,
with `SITE` as the batch variable and `Age`/`Sex` declared as covariates to preserve. The
supervised age-prediction models are then evaluated with the same Leave-One-Site-Out
cross-validation as before, on top of the harmonized features.

**What this does and does not leak:** the regression models (ElasticNet / RF / XGBoost) still
never see a held-out site's true age during their own training — LOSO-CV integrity for the
predictive model is intact. What *is* different from a strict out-of-sample design is that the
*harmonization* step (an unsupervised, population-level covariate-adjusted normalization, not
the predictive model itself) uses every subject's age — including held-out-fold subjects — to
estimate the shared, dataset-wide age–CT/FD relationship it should preserve while removing site
shifts. This is a mild, well-recognized simplification: we report it transparently here and
discuss it as a limitation in the report rather than implying a fully out-of-sample
harmonization we could not actually implement.

**Refinement found empirically: vanilla ComBat vs ComBat-GAM.** A first run with standard
(linear-covariate) ComBat *increased* MAE/RMSE and *decreased* pooled R² for every model,
compared to the raw (unharmonized) features — the opposite of the expected effect. The likely
cause: ComBat models the relationship between the preserved covariate (`Age`) and each feature
as **linear**, but the CT/FD–age relationship is demonstrably non-linear here (it is exactly
why ElasticNet trails the tree-based models throughout this notebook). Forcing a linear
preservation term while the true signal is curved lets ComBat distort real biological variance
along with the site effect it is meant to remove.

This is a documented limitation of standard ComBat, and the reason **ComBat-GAM** (Pomponio et
al., 2020 — reference #3) was introduced: it models continuous covariates with a smooth,
non-linear term (via B-splines / GAM) instead of a rigid linear coefficient. `neuroHarmonize`
supports this natively via the `smooth_terms` argument, used below for `Age`.

In [ ]:
HARM_FOLDS_PATH = f"{RESULTS_DIR}/loso_harmonized_folds.csv"
HARM_PREDS_PATH = f"{RESULTS_DIR}/loso_harmonized_preds.csv"

# Fit ComBat-GAM once, globally, on all 1,740 subjects (see markdown above for why global,
# and for why `smooth_terms=["Age"]` -- vanilla linear-covariate ComBat measurably hurt
# performance here, consistent with the known non-linear CT/FD-age relationship).
# Only the 22 CT/FD imaging features are harmonized -- Sex is a covariate, not an imaging
# measurement, so it is appended unchanged afterward to match MODEL_FEATURES' column order.
covars_full = df[["SITE", "Age", "Sex"]].copy()
_, X_harmonized_features = harmonizationLearn(
    df[FEATURES].values.astype(float), covars_full, smooth_terms=["Age"]
)
X_harmonized_full = np.hstack([X_harmonized_features, df[["Sex"]].values.astype(float)])
assert X_harmonized_full.shape[1] == len(MODEL_FEATURES)

if RUN_FULL_TRAINING or not os.path.exists(HARM_FOLDS_PATH):
    folds_harm, preds_harm = run_loso_cv(df, X_source=X_harmonized_full)
    folds_harm.to_csv(HARM_FOLDS_PATH, index=False)
    preds_harm.to_csv(HARM_PREDS_PATH, index=False)
else:
    folds_harm = pd.read_csv(HARM_FOLDS_PATH)
    preds_harm = pd.read_csv(HARM_PREDS_PATH)

folds_harm.groupby("model")[["mae", "rmse", "r2"]].mean().round(2)


In [ ]:
print("Pooled R^2 (all out-of-fold LOSO predictions combined), HARMONIZED features:")
for m in preds_harm["model"].unique():
    print(f"  {m:15s} R^2 = {pooled_r2(preds_harm, m):.3f}")


In [ ]:
all_folds = pd.concat([folds_raw, folds_harm], ignore_index=True)
all_folds["condition"] = all_folds["harmonized"].map({False: "Raw", True: "Harmonized"})

summary = (all_folds.groupby(["model", "condition"])[["mae", "rmse", "r2"]]
           .agg(["mean", "std"]).round(2))
summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=all_folds, x="model", y="mae", hue="condition",
            estimator=np.mean, errorbar="se", ax=ax)
ax.set_ylabel("Mean Absolute Error (years), across LOSO folds")
ax.set_title("Age prediction error: Raw vs Harmonized features\n"
             "(Leave-One-Site-Out cross-validation, out-of-sample ComBat)")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/mae_pre_post_harmonization.png", dpi=150)
plt.show()


In [ ]:
# Paired comparison (per-site MAE, Raw vs Harmonized) -- is the improvement significant?
for model_name in all_folds["model"].unique():
    raw_mae = (folds_raw[folds_raw.model == model_name]
               .set_index("site")["mae"].sort_index())
    harm_mae = (folds_harm[folds_harm.model == model_name]
                .set_index("site")["mae"].sort_index())
    common_sites = raw_mae.index.intersection(harm_mae.index)
    stat, p = stats.wilcoxon(raw_mae.loc[common_sites], harm_mae.loc[common_sites])
    direction = "lower (better)" if harm_mae.loc[common_sites].mean() < raw_mae.loc[common_sites].mean() else "higher"
    print(f"{model_name:15s} Wilcoxon p={p:.4f} | harmonized MAE is {direction} "
          f"({raw_mae.loc[common_sites].mean():.2f} -> {harm_mae.loc[common_sites].mean():.2f} years)")


In [ ]:
# Win/loss count per model: how many of the 36 sites individually improved (lower MAE)
# after harmonization vs. got worse -- an intuitive complement to the Wilcoxon p-value above,
# useful when the paired test lacks power (few, uneven-sized groups).
for model_name in all_folds["model"].unique():
    raw_mae = (folds_raw[folds_raw.model == model_name]
               .set_index("site")["mae"].sort_index())
    harm_mae = (folds_harm[folds_harm.model == model_name]
                .set_index("site")["mae"].sort_index())
    common_sites = raw_mae.index.intersection(harm_mae.index)
    diff = harm_mae.loc[common_sites] - raw_mae.loc[common_sites]
    n_improved = (diff < 0).sum()
    n_worsened = (diff > 0).sum()
    n_tied = (diff == 0).sum()
    print(f"{model_name:15s} improved: {n_improved:2d}/{len(common_sites)} sites | "
          f"worsened: {n_worsened:2d} | unchanged: {n_tied} | "
          f"median MAE change: {diff.median():+.2f} years")


## 10. Site Classifier AFTER Harmonization (Global Illustrative Check)

This is a standalone diagnostic (not part of the leakage-safe LOSO pipeline above): we harmonize the *entire* dataset once, globally, purely to visualize and quantify how much site information is removed. It answers a different question ("how separable are sites after harmonization, overall?") than the LOSO experiments above ("does harmonization help out-of-sample age prediction?").

In [ ]:
# Reuses X_harmonized_features (22 CT/FD columns, no Sex) computed in Section 8 --
# same globally-fit ComBat, this is just a different diagnostic on top of it.
bal_acc_post, macro_f1_post = site_classifier_cv(X_harmonized_features, sites)
print(f"Site classifier on RAW features        -- balanced accuracy: {bal_acc_pre:.3f} | macro-F1: {macro_f1_pre:.3f}")
print(f"Site classifier on HARMONIZED features -- balanced accuracy: {bal_acc_post:.3f} | macro-F1: {macro_f1_post:.3f}")


In [ ]:
pcs_harm = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(
    StandardScaler().fit_transform(X_harmonized_features)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
for ax, data_pcs, title in zip(axes, [pcs, pcs_harm], ["Raw features", "Harmonized features"]):
    ax.scatter(data_pcs[:, 0], data_pcs[:, 1], c=df["SITE"].astype("category").cat.codes,
               cmap="tab20", s=10, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("PC1")
axes[0].set_ylabel("PC2")
fig.suptitle("PCA colored by site — before vs after ComBat harmonization")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/pca_raw_vs_harmonized.png", dpi=150)
plt.show()


## 11. Model Interpretation

Feature importance / coefficients from models refit on the full (globally harmonized) dataset — purely for interpretation, not for the performance numbers reported above (those come exclusively from the leakage-safe LOSO pipeline).

In [ ]:
best_model_name = folds_harm.groupby("model")["mae"].mean().idxmin()
print("Best model (lowest mean LOSO MAE, harmonized):", best_model_name)

estimator, param_dist = get_model_space()[best_model_name]
final_model, final_params = tune_and_fit(estimator, param_dist, X_harmonized_full, df["Age"].values)
print("Selected hyperparameters:", final_params)

if hasattr(final_model, "feature_importances_"):
    importances = pd.Series(final_model.feature_importances_, index=MODEL_FEATURES)
elif hasattr(final_model, "coef_"):
    importances = pd.Series(np.abs(final_model.coef_), index=MODEL_FEATURES)

importances = importances.sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(8, 6))
importances.iloc[::-1].plot.barh(ax=ax)
ax.set_title(f"Top 15 feature importances — {best_model_name} (fit on harmonized data)")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/feature_importance.png", dpi=150)
plt.show()


## 12. Brain-Age Gap

Using the **out-of-fold** LOSO predictions from the harmonized pipeline (i.e. genuinely unseen-site predictions, not the interpretation refit above), we compute the brain-age gap = predicted age − chronological age. This is a well-studied biomarker in the literature (Cole & Franke, 2017) associated with neurodegenerative risk. **We cannot validate this claim on this dataset** — none of the source cohorts (ABIDE, FCP-ICBM, CoRR-NKI2, IXI) include neurodegenerative-disease diagnoses or an elderly clinical cohort — so this section is descriptive, discussed further as future work in the report.

In [ ]:
best_preds = preds_harm[preds_harm.model == best_model_name].copy()
best_preds["gap"] = best_preds["pred_age"] - best_preds["true_age"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(best_preds["true_age"], best_preds["pred_age"], s=10, alpha=0.5)
lims = [best_preds[["true_age", "pred_age"]].min().min(), best_preds[["true_age", "pred_age"]].max().max()]
axes[0].plot(lims, lims, "r--", lw=1, label="Identity")
axes[0].set_xlabel("Chronological age"); axes[0].set_ylabel("Predicted age")
axes[0].set_title(f"Predicted vs true age (out-of-fold, {best_model_name}, harmonized)")
axes[0].legend()

sns.histplot(best_preds["gap"], bins=40, ax=axes[1])
axes[1].axvline(0, color="r", ls="--", lw=1)
axes[1].set_xlabel("Brain-age gap (predicted − true, years)")
axes[1].set_title("Brain-age gap distribution")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/brain_age_gap.png", dpi=150)
plt.show()

print(f"Mean gap: {best_preds['gap'].mean():.2f} years | SD: {best_preds['gap'].std():.2f} years")


## Extension: Sex Classification (Same Framework, Different Target)

As a further test of the pipeline, we swap the target: predict `Sex` (binary: 0 = male,
1 = female) from the CT/FD imaging features, with `Age` now used as a predictor instead.
This reuses almost everything already built:

- The same Leave-One-Site-Out cross-validation structure (`SITE` held out, never seen in
  training or tuning).
- The same globally-fit ComBat-GAM harmonization from Section 8 — no need to re-harmonize,
  since `Sex` was already declared as a covariate to preserve in that original fit, alongside
  `Age`. `X_harmonized_features` (22 columns, no Sex/Age) is reused directly.
- A new, smaller set of classification models (Logistic Regression, Random Forest, XGBoost)
  and classification metrics (accuracy, balanced accuracy, F1, ROC AUC) in place of the
  regression ones.

We do not have a strong prior on the expected direction of the harmonization effect here
(unlike brain age, sex-related morphometric differences are not obviously non-linear in the
same way) — this section is exploratory, results are reported as observed.

In [ ]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

N_ITER_SEARCH_CLF = 20

def get_model_space_clf():
    return {
        "LogisticRegression": (
            LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
            {"C": np.logspace(-3, 2, 30)},
        ),
        "RandomForest": (
            RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
            {"n_estimators": [100, 200, 300], "max_depth": [3, 5, 8, 12, None],
             "min_samples_leaf": [1, 2, 4, 8], "max_features": ["sqrt", 0.5, 1.0]},
        ),
        "XGBoost": (
            XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss"),
            {"n_estimators": [100, 200, 300], "max_depth": [2, 3, 4, 6],
             "learning_rate": [0.01, 0.03, 0.05, 0.1],
             "subsample": [0.7, 0.85, 1.0], "colsample_bytree": [0.7, 0.85, 1.0]},
        ),
    }

def tune_and_fit_clf(estimator, param_dist, X_train, y_train):
    search = RandomizedSearchCV(
        estimator, param_distributions=param_dist, n_iter=N_ITER_SEARCH_CLF,
        cv=INNER_CV, scoring="balanced_accuracy",
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_params_


In [ ]:
def run_loso_cv_clf(data, feature_cols=None, X_source=None, target_col="Sex", verbose=True):
    # Leave-One-Site-Out CV for sex classification -- mirrors run_loso_cv (age regression)
    # but with classification models/metrics. When X_source is given, it reuses the same
    # globally-harmonized imaging features from Section 8 (no re-harmonization needed: Sex
    # was already a protected covariate in that fit, alongside Age).
    model_space = get_model_space_clf()
    fold_rows, pred_rows = [], []

    site_arr = data["SITE"].values
    y_full = data[target_col].values
    subj_full = data["Subject"].values
    X_full = X_source if X_source is not None else data[feature_cols].values.astype(float)
    harmonized = X_source is not None

    site_list = pd.unique(site_arr)
    for i, held_out in enumerate(site_list):
        train_mask = site_arr != held_out
        test_mask = ~train_mask

        X_train_use, X_test_use = X_full[train_mask], X_full[test_mask]
        y_train, y_test = y_full[train_mask], y_full[test_mask]
        test_subjects = subj_full[test_mask]

        for model_name, (estimator, param_dist) in model_space.items():
            best_model, best_params = tune_and_fit_clf(estimator, param_dist, X_train_use, y_train)
            y_pred = best_model.predict(X_test_use)
            y_proba = (best_model.predict_proba(X_test_use)[:, 1]
                       if hasattr(best_model, "predict_proba") else None)

            acc = accuracy_score(y_test, y_pred)
            multi_class_ok = len(np.unique(y_test)) > 1
            bal_acc = balanced_accuracy_score(y_test, y_pred) if multi_class_ok else np.nan
            f1 = f1_score(y_test, y_pred, zero_division=0)
            try:
                auc = roc_auc_score(y_test, y_proba) if (y_proba is not None and multi_class_ok) else np.nan
            except ValueError:
                auc = np.nan

            fold_rows.append({"site": held_out, "n_test": len(y_test), "model": model_name,
                               "accuracy": acc, "balanced_accuracy": bal_acc, "f1": f1,
                               "auc": auc, "harmonized": harmonized})
            for subj, ta, pa in zip(test_subjects, y_test, y_pred):
                pred_rows.append({"site": held_out, "model": model_name, "harmonized": harmonized,
                                   "subject": subj, "true_sex": ta, "pred_sex": pa})

        if verbose:
            print(f"[{i+1:2d}/{len(site_list)}] site={held_out:22s} n_test={len(y_test):4d}  done")

    return pd.DataFrame(fold_rows), pd.DataFrame(pred_rows)


In [ ]:
MODEL_FEATURES_SEX = FEATURES + ["Age"]

SEX_RAW_FOLDS_PATH = f"{RESULTS_DIR}/loso_sex_raw_folds.csv"
SEX_RAW_PREDS_PATH = f"{RESULTS_DIR}/loso_sex_raw_preds.csv"

if RUN_FULL_TRAINING or not os.path.exists(SEX_RAW_FOLDS_PATH):
    folds_sex_raw, preds_sex_raw = run_loso_cv_clf(df, feature_cols=MODEL_FEATURES_SEX)
    folds_sex_raw.to_csv(SEX_RAW_FOLDS_PATH, index=False)
    preds_sex_raw.to_csv(SEX_RAW_PREDS_PATH, index=False)
else:
    folds_sex_raw = pd.read_csv(SEX_RAW_FOLDS_PATH)
    preds_sex_raw = pd.read_csv(SEX_RAW_PREDS_PATH)

folds_sex_raw.groupby("model")[["accuracy", "balanced_accuracy", "f1", "auc"]].mean().round(3)


In [ ]:
SEX_HARM_FOLDS_PATH = f"{RESULTS_DIR}/loso_sex_harmonized_folds.csv"
SEX_HARM_PREDS_PATH = f"{RESULTS_DIR}/loso_sex_harmonized_preds.csv"

# Reuses X_harmonized_features (Section 8) -- just appends raw Age as an extra predictor,
# same way Sex was appended raw for the age-prediction task.
X_harmonized_sex = np.hstack([X_harmonized_features, df[["Age"]].values.astype(float)])
assert X_harmonized_sex.shape[1] == len(MODEL_FEATURES_SEX)

if RUN_FULL_TRAINING or not os.path.exists(SEX_HARM_FOLDS_PATH):
    folds_sex_harm, preds_sex_harm = run_loso_cv_clf(df, X_source=X_harmonized_sex)
    folds_sex_harm.to_csv(SEX_HARM_FOLDS_PATH, index=False)
    preds_sex_harm.to_csv(SEX_HARM_PREDS_PATH, index=False)
else:
    folds_sex_harm = pd.read_csv(SEX_HARM_FOLDS_PATH)
    preds_sex_harm = pd.read_csv(SEX_HARM_PREDS_PATH)

folds_sex_harm.groupby("model")[["accuracy", "balanced_accuracy", "f1", "auc"]].mean().round(3)


In [ ]:
all_folds_sex = pd.concat([folds_sex_raw, folds_sex_harm], ignore_index=True)
all_folds_sex["condition"] = all_folds_sex["harmonized"].map({False: "Raw", True: "Harmonized"})

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=all_folds_sex, x="model", y="balanced_accuracy", hue="condition",
            estimator=np.mean, errorbar="se", ax=ax)
ax.axhline(0.5, color="gray", ls="--", lw=1, label="Chance level")
ax.set_ylabel("Balanced accuracy, across LOSO folds")
ax.set_title("Sex classification: Raw vs Harmonized features\n(Leave-One-Site-Out cross-validation)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/sex_balanced_accuracy_pre_post.png", dpi=150)
plt.show()

summary_sex = (all_folds_sex.groupby(["model", "condition"])
               [["accuracy", "balanced_accuracy", "f1", "auc"]].agg(["mean", "std"]).round(3))
summary_sex


## 13. Reproducibility Notes

- Random seed fixed (`RANDOM_STATE = 42`) for NumPy, all model constructors, all CV splitters.
- All three models (ElasticNet, Random Forest, XGBoost) are deterministic given a fixed seed — no GPU non-determinism.
- Data are downloaded directly from the original Zenodo records (not redistributed), with the exact DOIs cited above.
- Code released under MIT license, GitHub repository archived on Zenodo (DOI badge in the repository README).
- `RUN_FULL_TRAINING` flag: set to `False` to reproduce this notebook's results from the cached CSVs in `results/` without re-running the full nested LOSO-CV.

In [ ]:
import sklearn, xgboost, neuroHarmonize
print("Environment snapshot")
print("-" * 40)
for name, mod in [("numpy", np), ("pandas", pd), ("scikit-learn", sklearn),
                   ("xgboost", xgboost)]:
    print(f"{name:15s} {mod.__version__}")
print(f"{'neuroHarmonize':15s} {getattr(neuroHarmonize, '__version__', 'n/a')}")
print(f"{'RANDOM_STATE':15s} {RANDOM_STATE}")


## References

1. Marzi, C., Giannelli, M., Barucci, A., Tessa, C., Mascalchi, M., Diciotti, S. (2024).
   *Efficacy of MRI data harmonization in the age of machine learning: a multicenter study
   across 36 datasets.* Scientific Data, 11, 115. https://doi.org/10.1038/s41597-023-02421-6
2. Fortin, J.-P. et al. (2018). *Harmonization of cortical thickness measurements across
   scanners and sites.* NeuroImage, 167, 104-120.
3. Pomponio, R. et al. (2020). *Harmonization of large multi-site imaging datasets for the
   analysis of brain imaging patterns throughout the lifespan.* NeuroImage, 208, 116450.
   (`neuroHarmonize` package)
4. Johnson, W.E., Li, C., Rabinovic, A. (2007). *Adjusting batch effects in microarray
   expression data using empirical Bayes methods.* Biostatistics, 8(1), 118-127. (ComBat)
5. Cole, J.H., Franke, K. (2017). *Predicting age using neuroimaging: innovative brain
   ageing biomarkers.* Trends in Neurosciences, 40(12), 681-690.
6. Di Martino, A. et al. (2014). *The Autism Brain Imaging Data Exchange: towards a
   large-scale evaluation of the intrinsic brain architecture in autism.* Mol Psychiatry.
7. Stark, P.B. (2018). *Before reproducibility must come preproducibility.* Nature, 557, 613.
8. Peng, R.D. (2011). *Reproducible research in computational science.* Science, 334(6060).
9. Carter, R.E., Attia, Z.I., Lopez-Jimenez, F., Friedman, P.A. (2019). *Pragmatic
   considerations for fostering reproducible research in artificial intelligence.*
   npj Digital Medicine, 2, 42.
